# Assignment 4 — Problem Formulation

**Course:** EPA141A Model Based Decision Making — Delft University of Technology  
**Model:** JUSTICE

---

## Learning Outcomes

1. Complete the **XLRM framework** for a climate policy problem under deep uncertainty.
2. Translate **justice principles** into formally specified measurable objectives.
3. Understand the **adaptive policy representation** (EMODPS with RBFs).
4. Explore the design space of possible ECR policies.
5. Formally structure a many-objective optimisation problem and articulate the assumptions embedded in it.

---

## Background

Problem formulation is the first step in structuring a many-objective optimisation problem. Before an MOEA can search for Pareto-optimal policies, we must decide *what* to optimise (objectives), *what* we can control (levers), *what* we cannot control (uncertainties), and *what* model relationships connect them. The **XLRM framework** — eXogenous uncertainties, Levers, Relationships, and Measures of performance — provides a systematic structure for this.

In JUSTICE, climate policies are represented as **adaptive emission control rate (ECR) pathways** generated by **radial basis functions (RBFs)**. Rather than pre-specifying a fixed time-schedule for emission reductions, RBFs map the current climate state (global temperature level and its rate of change) to an emission control rate for each of the 57 RICE50 regions. This **closed-loop** formulation means policies automatically adapt to how the climate actually evolves, rather than following a pre-planned calendar schedule — a key advantage under deep uncertainty.

The MOEA in Assignment 5 will search over the **RBF parameters** (centers, radii, and weights) to find combinations that simultaneously optimize multiple objectives. A critical design choice is the **social welfare function**: different aggregation methods (Utilitarian, Prioritarian, Sufficientarian, Egalitarian) encode different ethical principles about equity between regions, and each produces a different Pareto front. The problem formulation therefore embeds assumptions that directly shape the solution space explored in Assignment 5.

---


## Setup — Imports and model configuration

The cell below imports all required packages, configures the EMA Workbench logging level, and initialises the DataLoader and geometric constants (number of regions, timesteps, RBF architecture parameters) that are referenced throughout the assignment.  Run the following cell as is. 

In [2]:
import os, sys
# ── Add JUSTICE-main to sys.path so justice internal imports resolve ───────────
_here         = os.path.abspath('.')   # CWD = notebook dir in JupyterLab
_justice_root = os.path.normpath(os.path.join(_here, '../JUSTICE-main'))
_NOTEBOOK_DIR = _here

_PLOTS_DIR = os.path.join(_NOTEBOOK_DIR, "plots")
os.makedirs(_PLOTS_DIR, exist_ok=True)
if _justice_root not in sys.path:
    sys.path.insert(0, _justice_root)
os.chdir(_justice_root)

import warnings; warnings.filterwarnings("ignore")
import importlib, sys, os
import multiprocessing
try:
    multiprocessing.set_start_method('spawn')
except RuntimeError:
    pass  # already set
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from ema_workbench import (
    Model, RealParameter, IntegerParameter, CategoricalParameter,
    ScalarOutcome, ArrayOutcome,
    perform_experiments, ema_logging,
    SequentialEvaluator, MultiprocessingEvaluator,
)
from justice.model import JUSTICE
from justice.util.enumerations import WelfareFunction
from justice.util.data_loader import DataLoader
from justice.objectives.objective_functions import years_above_temperature_threshold

import matplotlib.path as _mpath

In [3]:
def _patched_path_deepcopy(self, memo=None):
    if memo is None: memo = {}
    new_path = _mpath.Path.__new__(_mpath.Path)
    memo[id(self)] = new_path
    verts = self._vertices.copy()
    codes = self._codes.copy() if self._codes is not None else None
    new_path.__init__(verts, codes,
                      _interpolation_steps=self._interpolation_steps, readonly=False)
    return new_path
_mpath.Path.__deepcopy__ = _patched_path_deepcopy

ema_logging.log_to_stderr(ema_logging.INFO)

N_REGIONS   = 57
N_TIMESTEPS = 286
N_RBFS      = 4
N_INPUTS    = 2

years   = np.arange(2015, 2301)
tau_hat = np.clip((years - 2015) / (2100 - 2015), 0, 1)
s_curve = 3 * tau_hat**2 - 2 * tau_hat**3

dl = DataLoader()

required = ["justice", "numpy", "pandas", "matplotlib", "ema_workbench", "scipy", "seaborn"]
for pkg in required:
    found = importlib.util.find_spec(pkg) is not None
    print(f"  {'✓' if found else '✗'}  {pkg}")
print(f"\nPython {sys.version.split()[0]}")
print(f"DataLoader ready — {len(dl.REGION_LIST)} regions, {N_TIMESTEPS} timesteps")

  ✓  justice
  ✓  numpy
  ✓  pandas
  ✓  matplotlib
  ✓  ema_workbench
  ✓  scipy
  ✓  seaborn

Python 3.12.13
DataLoader ready — 57 regions, 286 timesteps


---

## Step 1 — Complete the XLRM framework

**Task 1.1** — Add items to each XLRM category. 

**Task 1.2** — For each item you add, identify its EMA Workbench counterpart:
- Uncertainties (X) → `RealParameter` or `CategoricalParameter` on `em_model.uncertainties`
- Levers (L) → `RealParameter` on `em_model.levers`
- Measures (M) → `ScalarOutcome` on `em_model.outcomes`

This mapping defines the search space and the objective
space for the MOEA in Assignment 5.

In [4]:
#NOTE: The RBF architecture and objective functions are already configured for you
# If you want to experiment with the lever structure or objectives, go for it
# just make sure you can justify and properly implement every change.

xlrm = {
    "X — Uncertainties": [
        'rho: discount rate [0.001, 0.030]', 
        'eta: utility elasticity [0.5, 1.5]', 
        'delta: damage scale [0.5, 2.0]', 
        'ecs_ensemble: climate sensitivity [1, 1001]',
        "SSP-RCP scenario pair",
        "Welfare function (ethical aggregation rule)",
        # YOUR ANSWER HERE
    ],
    "L — Levers": [
        "RBF adaptive policy: 244 parameters (centers, radii, weights — searched by MOEA)",
    ],
    "R — Relationships (the model)": [
        "Emissions = GDP × carbon intensity × (1 − μ)",
        "FAIR climate model: emissions → atmospheric concentrations → temperature",
        "Kalkuhl damage function: temperature change → percentage GDP losses",
        "Enerdata MAC curves: abatement level μ → nonlinear abatement costs",
        "Ramsey growth model: savings rate balances present vs future consumption",
        "Economy-climate feedback loop: emissions, damages, and abatement interact dynamically",
        "Welfare aggregation over time using selected ethical welfare function",

    ],
    "M — Measures of performance": [
        "welfare: aggregate social welfare (scalar)",
        "years_above_temperature_threshold: years global mean > 2 °C ",
        "welfare_loss_damage: welfare loss due to climate damage",
        "welfare_loss_abatement: welfare loss due to abatement cost ",
    ],
}

df_xlrm = pd.DataFrame(
    [(cat, item) for cat, items in xlrm.items() for item in items],
    columns=["Component", "Item"]
)
with pd.option_context("display.max_colwidth", 90):
    display(df_xlrm)

# XLRM → EMA Workbench type mapping
#Complete the following..
ema_mapping = pd.DataFrame([
    #YOUR CODE HERE
    # ("X — Uncertainties", "YOUR CODE"),
    # ("L — Levers", "RBF centers c_i_j",  "RealParameter [-1, 1]", "em_model.levers"),
    # ("L — Levers", "RBF radii r_i_j",    "RealParameter [ 0, 1]", "em_model.levers"),
    # ("L — Levers", "RBF weights w_k_i",  "RealParameter [ 0, 1]", "em_model.levers")
    # ("M — Measures",      "years_above_temperature_threshold","ScalarOutcome MINIMIZE", "em_model.outcomes"),
     # X — uncertainties
    ("X — Uncertainties", "ρ (discount rate)", "RealParameter[0.001, 0.030]", "model.uncertainties"),
    ("X — Uncertainties", "η (utility elasticity)", "RealParameter[0.5, 1.5]", "model.uncertainties"),
    ("X — Uncertainties", "δ (damage scale)", "RealParameter[0.5, 2.0]", "model.uncertainties"),
    ("X — Uncertainties", "ECS (climate sensitivity)", "RealParameter[1, 1001]", "model.uncertainties"),
    ("X — Uncertainties", "SSP-RCP scenario pair", "CategoricalParameter", "model.uncertainties"),
    ("X — Uncertainties", "Welfare function", "CategoricalParameter", "model.uncertainties"),

    # L — levers
    ("L — Levers", "RBF centers (c1–c8)", "RealParameter[-1,1]", "model.levers"),
    ("L — Levers", "RBF radii (r1–r8)", "RealParameter[0,2]", "model.levers"),
    ("L — Levers", "RBF weights (w1–w228)", "RealParameter[-2,2]", "model.levers"),
    ("L — Levers", "ECR floor", "RealParameter[0,1]", "model.levers"),

    # M — outcomes
    ("M — Measures", "Welfare", "ScalarOutcome (minimize)", "model.outcomes"),
    ("M — Measures", "Years above 2°C", "ScalarOutcome (minimize)", "model.outcomes"),
    ("M — Measures", "Welfare loss from damages", "ScalarOutcome (minimize)", "model.outcomes"),
    ("M — Measures", "Welfare loss from abatement", "ScalarOutcome (minimize)", "model.outcomes"),
], columns=["XLRM", "Variable", "EMA type", "Assignment to"])

print("\nXLRM → EMA Workbench mapping:")
display(ema_mapping)

,Component,Item
0,X — Uncertainties,"rho: discount rate [0.001, 0.030]"
1,X — Uncertainties,"eta: utility elasticity [0.5, 1.5]"
2,X — Uncertainties,"delta: damage scale [0.5, 2.0]"
3,X — Uncertainties,"ecs_ensemble: climate sensitivity [1, 1001]"
4,X — Uncertainties,SSP-RCP scenario pair
5,X — Uncertainties,Welfare function (ethical aggregation rule)
6,L — Levers,"RBF adaptive policy: 244 parameters (centers, radii, weights — searched by MOEA)"
7,R — Relationships (the model),Emissions = GDP × carbon intensity × (1 − μ)
8,R — Relationships (the model),FAIR climate model: emissions → atmospheric concentrations → temperature
9,R — Relationships (the model),Kalkuhl damage function: temperature change → percentage GDP losses



XLRM → EMA Workbench mapping:


,XLRM,Variable,EMA type,Assignment to
0,X — Uncertainties,ρ (discount rate),"RealParameter[0.001, 0.030]",model.uncertainties
1,X — Uncertainties,η (utility elasticity),"RealParameter[0.5, 1.5]",model.uncertainties
2,X — Uncertainties,δ (damage scale),"RealParameter[0.5, 2.0]",model.uncertainties
3,X — Uncertainties,ECS (climate sensitivity),"RealParameter[1, 1001]",model.uncertainties
4,X — Uncertainties,SSP-RCP scenario pair,CategoricalParameter,model.uncertainties
5,X — Uncertainties,Welfare function,CategoricalParameter,model.uncertainties
6,L — Levers,RBF centers (c1–c8),"RealParameter[-1,1]",model.levers
7,L — Levers,RBF radii (r1–r8),"RealParameter[0,2]",model.levers
8,L — Levers,RBF weights (w1–w228),"RealParameter[-2,2]",model.levers
9,L — Levers,ECR floor,"RealParameter[0,1]",model.levers


## Step 2 — Justice principles

**Task 2.1** — Run JUSTICE under BAU (no abatement) with three welfare functions and compare results.

> **Key insight:** What changes
> is how welfare losses are aggregated, which determines the shape of the objective landscape
> the MOEA will search. The choice of welfare function therefore changes the Pareto front.

In [5]:
# ── Step 1: Define which welfare functions to compare ─────────────────────────
# Add at least 3 welfare functions to this dictionary.
# Key = a label you choose (used in the output table), Value = the WelfareFunction enum member.
# Available options: WelfareFunction.UTILITARIAN, PRIORITARIAN, SUFFICIENTARIAN, EGALITARIAN
# Tip: check justice/util/enumerations.py for the full list.

welfare_functions = {
    # "Utilitarian": WelfareFunction.UTILITARIAN,   ← example
    # YOUR CODE HERE
    "Utilitarian": WelfareFunction.UTILITARIAN,
    "Prioritarian": WelfareFunction.PRIORITARIAN,
    "Sufficientarian": WelfareFunction.SUFFICIENTARIAN,
    "Egalitarian": WelfareFunction.EGALITARIAN,
    # "Consumption_corridor_prioritarian": WelfareFunction.CONSUMPTION_CORRIDOR_PRIORITARIAN, 
    # "Consumption_corridor_utilitarian": WelfareFunction.CONSUMPTION_CORRIDOR_UTILITARIAN,
    # "Consumption_limitarian_prioritarian": WelfareFunction.LIMITARIAN_PRIORITARIAN, 
    # "Consumption_limitarian_utilitarian": WelfareFunction.LIMITARIAN_UTILITARIAN 
}

# ── Step 2: Define the BAU (no-abatement) policy ──────────────────────────────
# ecr_bau is a 2D array of zeros: shape (N_REGIONS, N_TIMESTEPS).
# Zero emission control rate = no abatement at all — business as usual.
# This is your baseline: no climate policy, just the economy running freely.

ecr_bau = np.zeros((N_REGIONS, N_TIMESTEPS))

# ── Step 3: Run the model for each welfare function and collect results ────────
# The loop is already structured for you. Your job is to fill in the JUSTICE
# initialization and the four scalar extractions inside the loop.
#
# Each iteration:
#   a) resets the singleton                → JUSTICE.hard_reset()
#   b) builds a fresh model               → JUSTICE(...)
#   c) simulates 2015–2300 under BAU      → model.run(...)
#   d) extracts all outputs as a dict     → model.evaluate()
#   e) computes four comparison scalars   → your code below
#   f) stores them under the wf label     → results_wf[wf_name]

results_wf = {}
for wf_name, wf in welfare_functions.items():

    JUSTICE.hard_reset()  # mandatory before every new JUSTICE() call

    model = JUSTICE(
        # YOUR CODE HERE — fill in the required parameters:
        start_year=2015,          # simulation start (hint: what year does JUSTICE begin?)
        end_year=2300,            # simulation end
        timestep=1,            # model integration step in years
        scenario=2,            # SSP-RCP index (0–7)
        climate_ensembles=1,   # how many FaIR members? keep it small here (1 is fine for speed)
        stochastic_run=True,      # True adds ensemble noise; False gives a deterministic run
        social_welfare_function=wf,  # already provided — this is what the loop varies
    )

    model.run(emission_control_rate=ecr_bau, endogenous_savings_rate=True) #check out justice/model.py
    datasets = model.evaluate()  # returns a dict: keys are variable names, values are numpy arrays
    #Hint: run print(datasets.keys()) to see all available keys
    # ── Extract scalar 1: aggregate welfare ───────────────────────────────────
    # datasets["welfare"] is an array; take the absolute value and convert to float.
    # Welfare is negative internally (higher magnitude = worse outcome).
    wf_val = float(np.abs(np.squeeze(datasets["welfare"]))) # YOUR CODE HERE

    # ── Extract scalar 2: years above 2°C ────────────────────────────────────
    # Use years_above_temperature_threshold(temperature_array, threshold).
    # Which key in datasets gives you global temperature?
    yat = float(np.squeeze(years_above_temperature_threshold(datasets["global_temperature"], 2.0))) # YOUR CODE HERE

    # ── Extract scalar 3: welfare loss from climate damages ───────────────────
    # Call model.welfare_function.calculate_welfare() with the damage cost array.
    # Pass welfare_loss=True to disable the sufficiency floor (it only applies to consumption).
    # The function returns 4 values; you only need the last one (the scalar aggregate).
    # The notation below uses _, _, _, varname = ... to discard the first three.
    _, _, _, wl_dam = model.welfare_function.calculate_welfare(
        datasets["damage_cost_per_capita"], welfare_loss=True)# YOUR CODE HERE

    # ── Extract scalar 4: welfare loss from abatement costs ───────────────────
    # Same pattern as above, but use abatement_cost_per_capita.
    _, _, _, wl_abt = model.welfare_function.calculate_welfare(
        datasets["abatement_cost_per_capita"], welfare_loss=True)# YOUR CODE HERE

    results_wf[wf_name] = {
        "welfare":                wf_val,
        "years_above_2C":         yat,
        "welfare_loss_damage":    float(np.abs(wl_dam)),
        "welfare_loss_abatement": float(np.abs(wl_abt)),
    }
    print(f"  {wf_name}: welfare={wf_val:.6f}, years>2°C={yat:.1f}")

# ── Step 4: Display the comparison table ──────────────────────────────────────
# After the loop, results_wf has one entry per welfare function.
# pd.DataFrame(results_wf).T turns it into a table: rows = welfare functions, columns = metrics.
df_wf = pd.DataFrame(results_wf).T.round(4)
print("\nComparison table:")
display(df_wf)

  Utilitarian: welfare=103.742429, years>2°C=258.0
  Prioritarian: welfare=414.948723, years>2°C=258.0
  Sufficientarian: welfare=104.209597, years>2°C=258.0
  Egalitarian: welfare=433.851034, years>2°C=258.0

Comparison table:


,welfare,years_above_2C,welfare_loss_damage,welfare_loss_abatement
Utilitarian,103.7424,258.0,3573.4569,74364.1321
Prioritarian,414.9487,258.0,24834.5974,318818.3307
Sufficientarian,104.2096,258.0,3573.4569,74364.1321
Egalitarian,433.8510,258.0,4633.8749,318818.3307


In [12]:
results_wf

{'Utilitarian': {'welfare': 103.74242850274561,
  'years_above_2C': 258.0,
  'welfare_loss_damage': 3573.4568940485447,
  'welfare_loss_abatement': 74364.13205076792},
 'Prioritarian': {'welfare': 414.9487231047606,
  'years_above_2C': 258.0,
  'welfare_loss_damage': 24834.59743423589,
  'welfare_loss_abatement': 318818.3307053328},
 'Sufficientarian': {'welfare': 104.2095969147095,
  'years_above_2C': 258.0,
  'welfare_loss_damage': 3573.4568940485447,
  'welfare_loss_abatement': 74364.13205076792},
 'Egalitarian': {'welfare': 433.8510338498596,
  'years_above_2C': 258.0,
  'welfare_loss_damage': 4633.874859764437,
  'welfare_loss_abatement': 318818.3307053328}}

In [11]:
help(model)

Help on JUSTICE in module justice.model object:

class JUSTICE(builtins.object)
 |  JUSTICE(*args, **kwargs)
 |
 |  This is the JUSTICE model.
 |
 |  This class implements the integrated assessment model JUSTICE.
 |  It performs extensive initialization by loading datasets, configuring climate components, setting up
 |  economic and damage sub-models, and initializing output data structures. The model is designed to run
 |  simulations either stepwise or over an entire time horizon and provides evaluation methods to assess
 |  welfare metrics and economic indicators.
 |
 |  Initialization Parameters:
 |      start_year (int): The beginning year for the simulation (default is 2015).
 |      end_year (int): The ending year for the simulation (default is 2300).
 |      timestep (int): The simulation timestep increment (default is 1).
 |      scenario (int): Identifier for the simulation scenario.
 |      climate_ensembles (int or list, optional): Specifies one or more climate ensemble ind

## Step 3 — Explore the ECR policy design space

**Task 3.1** — Plot 5 different ECR profiles. This illustrates that the MOEA must search
over an enormous space of possible emission pathways, not just a single scalar. This cell is to give you intuition about the search space, you can run it as is. 

> **Why this matters:** The RBF decision variables can produce any combination of
> these profiles — and infinitely many more — per region per year. The plots below
> capture only a tiny 1D slice of that space.

In [1]:
profiles = {
    "BAU (no abatement)"          : np.zeros(N_TIMESTEPS),
    "Low uniform (0.30)"          : 0.30 * s_curve,
    "High uniform (0.85)"         : 0.85 * s_curve,
    "Front-loaded (reduce early)" : np.clip(0.9 - 0.6 * tau_hat, 0, 1),
    "Back-loaded (delay action)"  : np.clip(1.8 * tau_hat - 0.5,  0, 1),
}

fig, ax = plt.subplots(figsize=(10, 5))
colors = ["#2196F3", "#4CAF50", "#F44336", "#FF9800", "#9C27B0"]
for (name, profile), color in zip(profiles.items(), colors):
    ax.plot(years, profile, label=name, color=color, linewidth=2.2)

ax.fill_between(years, 0, 1, alpha=0.04, color="grey")
ax.set_xlabel("Year", fontsize=11)
ax.set_ylabel("Emission Control Rate (ECR)", fontsize=11)
ax.set_title("Breadth of the ECR policy space — five example profiles", fontsize=12)
ax.legend(fontsize=9, loc="upper left")
ax.set_ylim(-0.05, 1.05)
ax.axhline(0, color="black", linewidth=0.5, linestyle="--")
ax.axhline(1, color="black", linewidth=0.5, linestyle="--")
plt.tight_layout()
display(fig)
plt.savefig(os.path.join(_PLOTS_DIR, "a04ema_ecr_profiles.png"), dpi=150, bbox_inches="tight")
print("Saved: a04ema_ecr_profiles.png")
plt.close(fig)

NameError: name 'np' is not defined

*What do you observe from the plot above?*

>your answer here

## Step 4 — Adaptive policies with RBFs: decision variable structure

**Task 4.1** — Calculate the exact number of decision variables.

**Task 4.2** — A fixed time-path ECR maps *year → action*: rigid, insensitive to
whether ECS turns out high or low. An RBF policy maps *current state → action*: if temperature
tracks below 2°C the RBF returns lower abatement; if overshooting it increases it automatically.
The decision variables encode a **decision rule**, not a **decision at every time step** — that is what
makes it adaptive (EMODPS principle).

In [7]:
n_rbfs    = N_RBFS     # 4  — number of radial basis functions
n_inputs  = N_INPUTS   # 2  — inputs to the RBF: current temperature, rate of temperature change
n_outputs = N_REGIONS  # 57 — one emission control rate (ECR) per RICE50 region

# Each RBF has three sets of parameters:
#   - Centers  (c): one value per RBF per input  → controls WHERE the basis function is centred
#   - Radii    (r): one value per RBF per input  → controls HOW WIDE the response is
#   - Weights  (w): one value per RBF per output → controls HOW MUCH each RBF contributes to each region's ECR
#
# Task: calculate the number of decision variables for each component, write the general formula
# Tip: think about the shape of each parameter array before writing the formula.

n_centers = n_rbfs*n_inputs # YOUR CODE HERE   — how many center values are there in total?
n_radii   =  n_rbfs*n_inputs # YOUR CODE HERE   — how many radius values are there in total?
n_weights =  n_rbfs*n_outputs # YOUR CODE HERE   — how many weight values are there in total?
n_total   = n_centers+n_radii+n_weights # YOUR CODE HERE   — sum of all three components

#print each component and the total number of decision variables.
print(f"{n_centers=}, {n_radii=}, {n_weights=}, {n_total=}") 

n_centers=8, n_radii=8, n_weights=228, n_total=244


## RBF adaptive policy

The helper below replicates the RBF formulation used by JUSTICE. Each basis function computes a Gaussian response to the current climate state (scaled temperature and warming rate), and the weighted sum of all basis functions determines the emission control rate prescribed to each of the 57 regions.

The demo parameters below are for illustration only — centers are sampled from [0, 1], radii from [0.05, 0.3]. The actual optimization in Assignment 5 searches over a wider space: centers ∈ [-1, 1] and radii ∈ [0, 1], allowing the MOEA to place basis functions anywhere in the (extended) state space and control how broadly or narrowly each one responds.

In [6]:
# ── What an RBF policy is ─────────────────────────────────────────────────────
# A Radial Basis Function (RBF) network is a mathematical policy that maps a
# state of the world to a policy action (emission control rate).
#
# Concretely: given the current scaled_temperature (how warm it is now) and
# scaled_difference (how fast temperature is changing since the last timestep),
# the RBF returns one ECR value per region — how much each region should abate.
#
# Instead of a fixed schedule like "always abate 50%", an RBF policy responds
# to what is actually happening. If warming is tracking low, it backs off.
# If the climate is overshooting, it ramps up. This is what makes it adaptive.
#
# The parameters (centers, radii, weights) shape that response — they are the
# decision variables the MOEA searches over in Assignment 5.

def rbf_policy(state_vector, centers, radii, weights):
    """
    state_vector : (n_inputs,)          — current scaled climate state
    centers      : (n_rbfs, n_inputs)   — where each basis function is centred
    radii        : (n_rbfs, n_inputs)   — how wide each basis function is
    weights      : (n_rbfs, n_outputs)  — how much each RBF contributes per region
    returns      : (n_outputs,)         — ECR per region, clipped to [0, 1]
    """
    phi = np.exp(-np.sum(((state_vector - centers) / (radii + 1e-6))**2, axis=1))
    weighted = weights * phi[:, np.newaxis]   # (n_rbfs, n_outputs)
    return np.clip(weighted.sum(axis=0), 0.0, 1.0)

# ── Reflect ───────────────────────────────────────────────────────────────────
# Q1: A fixed time-path policy maps year → ECR. An RBF maps climate state → ECR.
#     Why is the second formulation preferable when climate sensitivity is uncertain?
# Q2: The inputs here are scaled temperature and warming rate.
#     What other state variables could you add, and what behaviour would they enable?

## Step 5 — Define the multi-objective optimisation problem

This step translates the XLRM framework and RBF policy structure from the previous steps into a concrete optimisation configuration. You will make four decisions that together define the problem:

1. **Social welfare function** — whose preferences does the model represent?
2. **Reference SSP-RCP scenario** — under what emissions pathway(s) do you optimise?
3. **FaIR ensemble strategy** — how do you represent climate uncertainty efficiently?
4. **Objectives and ε-resolution** — what do you optimise, in which direction, and how finely?

The result is saved as `config/config_student.json` inside `JUSTICE-main/`. Assignment 5 loads this file directly to configure every aspect of the MOEA run.

### Task 5.1 — Choose a social welfare function

JUSTICE supports multiple social welfare functions (you explored them in Step 2). \
The MOEA will optimise the 244 RBF parameters to minimise welfare loss as measured \
by whichever function you select here.

**Question:** Which welfare function should the reference optimisation use, and why?

In [7]:

# List the welfare functions you want to consider for your optimisation.
# You explored their differences in Step 2 — use that to justify your choice here.
# Format: "Label" : WelfareFunction.ENUM_MEMBER
# Available: UTILITARIAN, PRIORITARIAN, SUFFICIENTARIAN, EGALITARIAN

WELFARE_OPTIONS = {
    # YOUR CODE HERE
    # "Utilitarian": WelfareFunction.UTILITARIAN,  ← example
    "Utilitarian": WelfareFunction.UTILITARIAN,
    "Prioritarian": WelfareFunction.PRIORITARIAN,
    "Sufficientarian": WelfareFunction.SUFFICIENTARIAN,
    "Egalitarian": WelfareFunction.EGALITARIAN,
}

print("Welfare functions considered:")
for name, wf in WELFARE_OPTIONS.items():
    print(f"  {name:<20s}  internal index = {wf.value[0]}")

# ── Choose one for the optimisation ──────────────────────────────────────────
# From the options above, pick the one your MOEA will optimise under.
# Store it in CHOSEN_WELFARE_FUNCTION — this variable is used in Task 5.5 to build the config.

CHOSEN_WELFARE_FUNCTION     = WELFARE_OPTIONS['Sufficientarian'] # YOUR CODE HERE  
CHOSEN_WELFARE_FUNCTION_IDX = CHOSEN_WELFARE_FUNCTION.value[0]  # index used in the config file

print(f"\nChosen: {CHOSEN_WELFARE_FUNCTION.name}  (index {CHOSEN_WELFARE_FUNCTION_IDX})")
print("Rationale: LDCs tend to be lower income and more climate-vulnerable. Sufficientarian welfare function benefits the worst-off.")

Welfare functions considered:
  Utilitarian           internal index = 0
  Prioritarian          internal index = 1
  Sufficientarian       internal index = 2
  Egalitarian           internal index = 3

Chosen: SUFFICIENTARIAN  (index 2)
Rationale: LDCs tend to be lower income and more climate-vulnerable. Sufficientarian welfare function benefits the worst-off.


### Task 5.2 — Choose the reference SSP-RCP scenario

The MOEA evaluates each candidate policy under a **reference scenario**. \
JUSTICE supports eight SSP-RCP pathways (indices 0–7).

**Question:** Which scenario(s) should the reference optimisation use, and why?


RCP = Representative Concentration Pathways \
SSP = Shared Socioeconomic reference Pathways

In [14]:
# YOUR ANSWER HERE:
# SSP-RCP scenarios supported by JUSTICE 
# Source: justice/util/enumerations.py — Scenario class
# from justice.util.enumerations import Scenario

# # Extract names and indices
# scenarios = {s.name: s for s in Scenario}
# referencescenario = scenarios['SSP245']
# CHOSEN_SCENARIO_INDEX = referencescenario.value[0]

from justice.util.enumerations import Scenario

print("SSP-RCP scenarios supported by JUSTICE:")
for s in Scenario:
    print(f"  index {s.value[0]}  {s.name}")

CHOSEN_SCENARIO_INDEX = 5   # SSP3-7.0 -- regional rivalry, high emissions, weak cooperation

print(f"\nChosen reference scenario: index {CHOSEN_SCENARIO_INDEX} (SSP3-7.0)")
print(
    "Rationale: SSP3-7.0 is the most stressful plausible-but-not-extreme pathway for \n"
    "LDCs -- fragmented cooperation, slow tech diffusion, high inequality, and warming \n"
    "well above 2C by 2100. Optimising under this scenario forces the MOEA to find \n"
    "adaptive policies that protect the sufficiency floor for rsas and rsaf even when \n"
    "global cooperation breaks down. Robustness across all 8 scenarios is tested in A7."
)

SSP-RCP scenarios supported by JUSTICE:
  index 0  SSP119
  index 1  SSP126
  index 2  SSP245
  index 3  SSP370
  index 4  SSP434
  index 5  SSP460
  index 6  SSP534
  index 7  SSP585

Chosen reference scenario: index 5 (SSP3-7.0)
Rationale: SSP3-7.0 is the most stressful plausible-but-not-extreme pathway for 
LDCs -- fragmented cooperation, slow tech diffusion, high inequality, and warming 
well above 2C by 2100. Optimising under this scenario forces the MOEA to find 
adaptive policies that protect the sufficiency floor for rsas and rsaf even when 
global cooperation breaks down. Robustness across all 8 scenarios is tested in A7.


### Task 5.3 — Select FaIR climate ensemble members

JUSTICE runs with 1001 FaIR ensemble members.  The 1001 ensemble members represent uncertainty about how the physical climate system responds to CO₂. Each member is a different configuration of climate parameters, most importantly equilibrium climate sensitivity (ECS), which is how much global temperature rises when CO₂ doubles.

Nobody knows the exact value of ECS. Scientists estimate it's somewhere between ~2.5°C and ~4°C, but there's quite a bit of uncertainty. Rather than picking one value and pretending it's correct, FaIR runs 1001 plausible versions of the climate system simultaneously.

So when JUSTICE outputs fraction_above_threshold:

if a policy where to run through all 1001 climate versions, it
counts how many of them exceed 2°C
reports that fraction (e.g. 0.4 means 400 out of 1001 members crossed 2°C)
You can imagine that the same emission policy can lead to different temperature outcomes depending on which version of the climate turns out to be correct.\
Using all 1001 members per function evaluation is accurate but makes a 50 000-NFE run \
take weeks on a laptop. So you need to make a well-justified pragmatic choice.  You are\
of course welcome to explore DelftBlue HPC to do the run across the entire ECS ensemble. 

**Important:** FaIR ensemble indices are **not ordered by ECS** — index 5 is not \
necessarily warmer than index 4. The indices are identifiers, not a ranking.

**How to pass ensemble members to JUSTICE:**

climate_ensembles accepts either a single integer or a list of integers:
```python
JUSTICE(climate_ensembles=[0, 250, 500, 750, 1000])                   # hand-picked subset
JUSTICE(climate_ensembles=list(np.linspace(0, 1000, 10, dtype=int)))  # evenly-spaced subset of 10
JUSTICE(climate_ensembles= ensemble_indices)  # the ensemble_indices will be defined by you below. 
```

**Question:** How many ensemble members should a local optimisation run use, and how \
should they be selected?

> The full 1001-member would be very doable for HPC runs, check out: https://www.tudelft.nl/dhpc/system.

In [13]:
# ── YOUR TASK ──────────────────────────────────────────────────────────────────
# 1. Choose how many ensemble members to use for local optimisation.
#    Store this in N_ENSEMBLE_LOCAL.
#
# 2. Build a list of ensemble indices called ensemble_indices.
#    - Valid indices are 0–1000
#    - Since indices are not ordered by ECS, how do you select your sample? 
# 3. Print:
#    - The selected indices
#    - The approximate speedup vs the full 1001-member ensemble
#
# 4. Justify your choice in one sentence: why did you pick this number of members, and how?

FULL_ENSEMBLE_SIZE = 1001
N_ENSEMBLE_LOCAL   = 10
# JUSTICE uses 1-based ensemble indexing internally; sample evenly across the range.
ensemble_indices   = list(np.linspace(1, 1000, N_ENSEMBLE_LOCAL, dtype=int))

speedup = FULL_ENSEMBLE_SIZE / N_ENSEMBLE_LOCAL
print(f"Selected {N_ENSEMBLE_LOCAL} FaIR ensemble members:")
print(f"  indices: {ensemble_indices}")
print(f"  approximate speedup vs full ensemble: ~{speedup:.0f}x")
print(
    "\nJustification: 10 evenly-spaced indices give broad coverage of the ECS distribution \n"
    "(indices are identifiers, not a ranking, so stratified sampling across the index range \n"
    "is a reasonable proxy for stratified sampling across ECS). This keeps a 5-seed local \n"
    "MOEA run within a ~25-minute laptop budget while still preserving meaningful spread in \n"
    "fraction_above_threshold. The full 1001-member ensemble is reserved for HPC or for \n"
    "robustness re-evaluation of the final Pareto front in A7."
)

Selected 10 FaIR ensemble members:
  indices: [np.int64(1), np.int64(112), np.int64(223), np.int64(334), np.int64(445), np.int64(556), np.int64(667), np.int64(778), np.int64(889), np.int64(1000)]
  approximate speedup vs full ensemble: ~100x

Justification: 10 evenly-spaced indices give broad coverage of the ECS distribution 
(indices are identifiers, not a ranking, so stratified sampling across the index range 
is a reasonable proxy for stratified sampling across ECS). This keeps a 5-seed local 
MOEA run within a ~25-minute laptop budget while still preserving meaningful spread in 
fraction_above_threshold. The full 1001-member ensemble is reserved for HPC or for 
robustness re-evaluation of the final Pareto front in A7.


### Task 5.4 — Define the objectives and ε-resolution

JUSTICE returns four scalar outputs per policy evaluation. You must decide:
- Which to include as **optimisation objectives**
- The **direction** (minimise or maximise)
- The **ε value** — the Pareto archive resolution; smaller ε = finer front 

The four outputs and their typical scales are:

| Output | Typical scale | Meaning |
|---|---|---|
| `welfare` | ~10²–10⁵ | Absolute welfare loss (Utilitarian aggregation) |
| `fraction_above_threshold` | 0 – 1 | Fraction of FaIR members where global T > 2°C in 2100 |
| `welfare_loss_damage` | ~10³–10⁴ | Welfare burden attributable to climate damages |
| `welfare_loss_abatement` | ~10³–10⁴ | Welfare burden attributable to abatement costs |

**Question:** What direction should each objective have and what ε-value is appropriate?

Tips:
> All four objectives are physically *lower-is-better*. However, the JUSTICE model computes `welfare_loss_damage` and `welfare_loss_abatement` as **positive magnitudes** (`np.abs()` of the welfare function output). This may seem counterintuitive: The welfare function applied to small positive costs returns a large negative number, so np.abs is inversely related to actual cost — larger = less damage)

> ε should be proportional to the objective's scale: 0.1 and 0.25 give fine resolution over the 0–1 ranges; 10.0 gives ~0.1–1% resolution over the 10³–10⁴ welfare loss range.

In [15]:
from ema_workbench import ScalarOutcome

# EPSILONS = [
#     2000,  # welfare
#     0.05,  # fraction above threshold
#     1000,  # welfare loss damage
#     1000   # welfare loss abatement
#     # YOUR CODE HERE — four values, one per objective, the order matters!
# ]

EPSILONS = [10.0, 0.05, 10.0, 10.0]

objectives = [
    ScalarOutcome("welfare",                      kind=ScalarOutcome.MINIMIZE),
    ScalarOutcome("fraction_above_threshold",     kind=ScalarOutcome.MINIMIZE),
    ScalarOutcome("welfare_loss_damage",          kind=ScalarOutcome.MAXIMIZE),
    ScalarOutcome("welfare_loss_abatement",       kind=ScalarOutcome.MAXIMIZE)
#YOUR CODE HERE Set up the objectives, direction of optimization, and epsilon values
]

#PRINT YOUR SPECIFICATIONS
print("Objectives and epsilon-resolution:")
print(f"  {'name':<28s}  {'direction':<10s}  epsilon")
for obj, eps in zip(objectives, EPSILONS):
    direction = "MINIMIZE" if obj.kind == ScalarOutcome.MINIMIZE else "MAXIMIZE"
    print(f"  {obj.name:<28s}  {direction:<10s}  {eps}")

Objectives and epsilon-resolution:
  name                          direction   epsilon
  welfare                       MINIMIZE    10.0
  fraction_above_threshold      MINIMIZE    0.05
  welfare_loss_damage           MAXIMIZE    10.0
  welfare_loss_abatement        MAXIMIZE    10.0


### Task 5.5 — Assemble and save the optimisation configuration

Collect all four decisions into a single JSON config file. `run_optimization_local.py` (Assignment 5) reads this file to set up every run parameter — welfare function, scenario, ensemble size, RBF architecture, objectives, and ε-values.

In [ ]:
import json, os

# ── Path setup (do not modify) ────────────────────────────────────────────────
try:
    _NOTEBOOK_DIR = os.path.dirname(os.path.abspath(__vsc_ipynb_file__))
except NameError:
    _NOTEBOOK_DIR = os.path.abspath('.')
_CONFIG_DIR = os.path.join(_NOTEBOOK_DIR, "../config")
os.makedirs(_CONFIG_DIR, exist_ok=True)

# ── Assemble config from your decisions in Tasks 5.1–5.4 ─────────────────────
# Fill in each value. CHOSEN_SCENARIO_INDEX and EPSILONS should already be
# defined in the cells above — you can reference them directly. 

student_config = {
    "start_year": 2015                       ,  # YOUR CODE HERE
    "end_year":  2300                      ,  # YOUR CODE HERE
    "data_timestep": 5                   ,  # years between raw SSP input data points
    "timestep": 1                        ,  # model integration step in years

    # First year the MOEA is allowed to set ECR > 0
    "emission_control_start_year":  2050    ,  # YOUR CODE HERE 

    # RBF architecture — n_rbfs = n_inputs + 2 is the JUSTICE rule of thumb
    "n_rbfs":  N_RBFS                          ,  # YOUR CODE HERE 
    "n_inputs":  N_INPUTS                      ,  # YOUR CODE HERE (you can use N_INPUTS)

    # Epsilon values from Task 5.4 — one per objective, order matters
    "epsilons":  EPSILONS                       ,  # YOUR CODE HERE (use EPSILONS),

    # Year at which fraction_above_threshold is evaluated
    "temperature_year_of_interest":  2100   ,  # YOUR CODE HERE

    # SSP-RCP index from Task 5.2
    "reference_ssp_rcp_scenario_index": CHOSEN_SCENARIO_INDEX,
    "welfare_function":                 CHOSEN_WELFARE_FUNCTION.name,
}

# ── Validation (do not modify) ────────────────────────────────────────────────
assert student_config["n_rbfs"] == student_config["n_inputs"] + 2, \
    "n_rbfs must equal n_inputs + 2"
assert len(student_config["epsilons"]) == 4, \
    "Exactly 4 epsilon values required (one per objective)"
assert all(e > 0 for e in student_config["epsilons"]), \
    "All epsilon values must be strictly positive"
assert student_config["emission_control_start_year"] >= student_config["start_year"], \
    "Policy cannot start before the simulation"

# ── Save ──────────────────────────────────────────────────────────────────────
config_path = os.path.join(_CONFIG_DIR, "config_student.json")
with open(config_path, "w") as fh:
    json.dump(student_config, fh, indent=4)

print(f"Config saved → {config_path}")
print()
print(json.dumps(student_config, indent=4))


Config saved → c:\Users\leahm\.vscode\EPA141\LDC_Group-\assignments_ema\../config\config_student.json

{
    "start_year": 2015,
    "end_year": 2300,
    "data_timestep": 5,
    "timestep": 1,
    "emission_control_start_year": 2050,
    "n_rbfs": 4,
    "n_inputs": 2,
    "epsilons": [
        10.0,
        0.05,
        10.0,
        10.0
    ],
    "temperature_year_of_interest": 2200,
    "reference_ssp_rcp_scenario_index": 5,
    "welfare_function": "SUFFICIENTARIAN"
}


---
## Reflection Questions 
**1. Single-scenario optimisation.** The MOEA optimises under SSP2-RCP4.5 alone. What risk does this introduce?

**2. Closed-loop vs. open-loop policies.** The RBF encodes a *decision rule* (react to current climate state) rather than a *decision schedule* (follow a fixed time path). Why is the closed-loop formulation preferable when climate sensitivity is uncertain? What is the computational cost of this flexibility?

**3. FaIR ensemble trade-off.** There is a trade-off between computational cost and the representation of climate uncertainty in the optimisation. How does your choice of ensemble size affect the results, and what would you need to check to have confidence in your choice?
